# encoder
# decoder
# train
# generate

In [1]:
import torch
import torch.nn as nn

In [2]:
chars = "0123456789-/"
special_tokens = ["<SOS>", "<EOS>"]

itos = special_tokens + list(chars)
stoi = {ch: i for i, ch in enumerate(itos)}

SOS_token = stoi["<SOS>"]
EOS_token = stoi["<EOS>"]

vocab_size = len(itos)

print(itos)
print(stoi)

['<SOS>', '<EOS>', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '-', '/']
{'<SOS>': 0, '<EOS>': 1, '0': 2, '1': 3, '2': 4, '3': 5, '4': 6, '5': 7, '6': 8, '7': 9, '8': 10, '9': 11, '-': 12, '/': 13}


In [3]:
class EncoderRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.1)

        # 你来写 embedding
        # 你来写 rnn

    def forward(self, input):
        # 要限定 input的shape
        # input.shape (batch_size, seqlen)
        # 你来写一次前向传播
        input = self.embedding(input)
        input = self.dropout(input)
        # hidden不受 batch_first=True的控制，其shape仍然为(direction*num,batch_size,hidden_size)
        # output.shape 是 (batch_size,seq_len,direction*hidden_size)
        output, hidden = self.rnn(input)
        return output, hidden


In [4]:
import random
def make_sample():
    ep = 100
    inputs = []
    targets = []
    for i in range(ep):
        year = random.randint(1950, 2050)
        day = random.randint(1, 28)
        month = random.randint(1, 12)
        input = f"{year}-{month}-{day}"
        inputs.append(input)
        target = f"{day}/{month}/{year}"
        targets.append(target)
    return inputs, targets

inputs, targets = make_sample()
print(inputs[0:10])
print(targets[0:10])


['1980-1-10', '2036-10-9', '1992-3-15', '2001-12-18', '2031-8-28', '1989-7-26', '2043-5-26', '1991-3-2', '2018-7-3', '1986-5-18']
['10/1/1980', '9/10/2036', '15/3/1992', '18/12/2001', '28/8/2031', '26/7/1989', '26/5/2043', '2/3/1991', '3/7/2018', '18/5/1986']


In [5]:
def tensor_from_string(s, add_eos=False):
    input_tensor = torch.tensor([stoi[ch] for ch in s], dtype=torch.long)
    if add_eos:
        new_value = torch.tensor([EOS_token])
        input_tensor = torch.cat([input_tensor, new_value],dim=0)
    return input_tensor.unsqueeze(0)

def string_from_tensor(tensor):
    s = ''
    tensor = tensor.squeeze(0)
    for token in tensor:
        idx = token.item()
        if idx != EOS_token:
            s += itos[idx]
    return s

In [6]:

x = tensor_from_string("2002-1-23")
print(x, x.shape)
y = tensor_from_string("23/1/2002", add_eos=True)
print(y, y.shape)


tensor([[ 4,  2,  2,  4, 12,  3, 12,  4,  5]]) torch.Size([1, 9])
tensor([[ 4,  5, 13,  3, 13,  4,  2,  2,  4,  1]]) torch.Size([1, 10])


In [7]:
print(string_from_tensor(x))
print(string_from_tensor(y))

2002-1-23
23/1/2002


In [8]:
hidden_size = 5
encoder = EncoderRNN(vocab_size, hidden_size)

x = tensor_from_string("2002-1-23")
print("x", x.shape)

encoder_output, encoder_hidden = encoder.forward(x);
print("encoder_output:", encoder_output.shape)
print("encoder_hidden:", encoder_hidden.shape)

x torch.Size([1, 9])
encoder_output: torch.Size([1, 9, 5])
encoder_hidden: torch.Size([1, 1, 5])


In [9]:
from torch import Tensor


class DecoderRNN(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int, output_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_hidden: Tensor, target_tensor: Tensor, max_len: int):
        # encoder_output是encoder输出的 hiddeng向量
        # target_tensor.shape is (batch_size, seq_len)
        # foward.shape is (batch_size, seq_len,vocab_size)
        batch_size = encoder_hidden.shape[1]
        hidden = encoder_hidden
        input_token = SOS_token 
        outputs = []
        # input_token的格式是 (batch_size,seq_len)
        input_token = torch.full((batch_size, 1), SOS_token, dtype=torch.long) #设置起始词元
        loop_len = max_len
        if target_tensor is not None:
            loop_len = target_tensor.shape[1]

        for i in range(loop_len):
            hidden, logits, predictIdx = self.forward_step(hidden, input_token) 
            outputs.append(logits)

            if target_tensor is not None:
                input_token = target_tensor[:,i].unsqueeze(1)
            else :
                input_token = predictIdx.detach()
        decoder_outputs = torch.stack(outputs, dim=1)
        return decoder_outputs, hidden
    
    def forward_step(self, hidden, input_token):
        # input_token.shape是（batch_size,seq_len)
        embedded = self.embedding(input_token)
        # rnn的hidden输入是(batch_size,seq_len,hidden)
        output, hidden = self.rnn(embedded, hidden)
        # hidden不受 batch_first=True的控制，其shape仍然为(direction*num,batch_size,hidden_size)
        # output.shape 是 (batch_size,seq_len,direction*hidden_size)
        logits = self.out(hidden[-1]) # hidden[-1]表示最后一个num
        predictIdx = logits.argmax(dim = -1,keepdim= True)
        return logits, hidden, predictIdx

    

In [23]:
hidden_size = 5
encoder = EncoderRNN(vocab_size, hidden_size)
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)
logits, decoder_hidden, predictIdx = decoder.forward_step(encoder_hidden, input_token)

print("decoder_outputs:", decoder_outputs.shape)
print("decoder_hidden:", decoder_hidden.shape)
print("target:", y.shape)


decoder_outputs: torch.Size([1, 10, 14])
decoder_hidden: torch.Size([1, 1, 5])
target: torch.Size([1, 10])
